# 01. Случайно собрать фотографии из iNaturalist

**Входы:** только `sampling_plan.json` и конкурсные `dataset.jsonl` + `submission-template.jsonl`. Ноутбук обращается к публичному API iNaturalist, набирает кандидатов по группам из плана, случайно выбирает 400 основных и 100 отдельных фото для материалов исполнителей, проверяет их оригиналы и записывает два JSONL.

**Выходы в `sampling_runs/<время запуска>/`:**

- `main_dataset_draft.jsonl` — 400 фото для bbox → review → сравнения моделей;
- `worker_training_exam_draft.jsonl` — 100 фото для будущих обучения, экзамена и контроля;
- `candidates_from_api.jsonl`, `selection_500_proposal.csv`, `candidate_gallery_500.html`, `download_audit.jsonl`, `similarity_review.csv`, `sampling_audit.json` — данные для просмотра выборки и проверки отбора.

Таблица отбора и HTML-галерея помогают просмотреть выборку перед разметкой. Оригиналы сохраняются в `images/`.

Выбор случайный **среди наблюдений с указанных в плане страниц API**, а не равномерный среди всего iNaturalist. Зафиксированный `seed` делает выбор воспроизводимым при одинаковом ответе API; сам iNaturalist со временем меняется. Категория наблюдения не является меткой видимого объекта. До платных заданий нужно просмотреть галерею и утвердить итоговый список. Ноутбук не создаёт проекты или пулы Яндекс Заданий.

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import csv
import hashlib
import json
import re
import time

root = Path.cwd()
plan_path = root / "sampling_plan.json"
contest_path = root / "dataset.jsonl"
contest_template_path = root / "submission-template.jsonl"
for path in (plan_path, contest_path, contest_template_path):
    if not path.is_file():
        raise FileNotFoundError(f"Не найден {path.name}; открой ноутбук из папки проекта")

plan = json.loads(plan_path.read_text(encoding="utf-8"))
groups = plan["groups"]
seed = plan["seed"]
if type(seed) is not int or not groups or len({g["name"] for g in groups}) != len(groups):
    raise ValueError("Проверь seed и уникальность названий групп в sampling_plan.json")
if sum(g["target_count"] for g in groups) != 400:
    raise ValueError("В плане должно быть ровно 400 основных фото")
if sum(g["training_exam_count"] for g in groups) != 100:
    raise ValueError("В плане должно быть ровно 100 фото для материалов исполнителей")
candidate_total = sum(g["candidate_count"] for g in groups)

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def write_jsonl(path, rows):
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8") as target:
        for row in rows:
            target.write(json.dumps(row, ensure_ascii=False) + "\n")
    temporary.replace(path)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def photo_map(path):
    rows = read_jsonl(path)
    result = {}
    for row in rows:
        photo_id, url = row.get("photo_id"), row.get("photo_url")
        if type(photo_id) is not int or not isinstance(url, str) or not url:
            raise ValueError(f"{path.name}: некорректные photo_id/photo_url")
        if photo_id in result:
            raise ValueError(f"{path.name}: повторный ID {photo_id}")
        result[photo_id] = url
    if len(result) != 400 or len(set(result.values())) != 400:
        raise ValueError(f"{path.name}: нужны ровно 400 уникальных ID и URL")
    return result

contest = photo_map(contest_path)
if photo_map(contest_template_path) != contest:
    raise ValueError("Шаблон Контеста и dataset.jsonl содержат разные фотографии")

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
run_dir = root / "sampling_runs" / run_stamp
if run_dir.exists():
    raise FileExistsError(f"Папка запуска уже существует: {run_dir}")
run_dir.mkdir(parents=True)
retrieved_at = datetime.now(timezone.utc).isoformat()
print("Запуск:", run_dir)
print("План: seed =", seed, "; кандидатов =", candidate_total)
print("Конкурсных ID:", len(contest))

## 1. Получить кандидатов через API

Запросы идут по `taxon_id`, `without_taxon_id`, `pages` и `candidate_count` из плана. У каждого фото отдельно проверяется `license_code=cc0`; конкурсные `photo_id` исключаются ещё до случайного выбора. Между запросами есть пауза около секунды.

In [ ]:
from scripts.collect_candidates import collect_group

candidate_rows = []
seen_ids = set()
for index, group in enumerate(groups):
    if index:
        time.sleep(1.1)
    group_rows = collect_group(
        group=group,
        exam_ids=set(contest),
        seen_ids=seen_ids,
        seed=seed + index,
        retrieved_at=retrieved_at,
        dry_run=False,
    )
    candidate_rows.extend(group_rows)
if len(candidate_rows) != candidate_total or len({r["photo_id"] for r in candidate_rows}) != candidate_total:
    raise ValueError("Не удалось собрать нужное число уникальных кандидатов")
if any(r["license_code"] != "cc0" or r["photo_id"] in contest for r in candidate_rows):
    raise ValueError("Найден кандидат без CC0 или с конкурсным ID")
write_jsonl(run_dir / "candidates_from_api.jsonl", candidate_rows)
print("Собрано кандидатов:", len(candidate_rows))

## 2. Случайно выбрать 400 + 100

Из полученных кандидатов оставляются фото, у которых меньшая сторона оригинала не короче 400 пикселей. В каждой группе основная и учебная квоты выбираются случайно с зафиксированным `seed`. Для просмотра выборки создаются `selection_500_proposal.csv` и `candidate_gallery_500.html`.

In [ ]:
from scripts.prepare_review import suggest, write_csv, write_gallery

chosen = suggest(candidate_rows, plan)
if len(chosen) != 500:
    raise ValueError(f"Ожидалось 500 выбранных фото, получено {len(chosen)}")
selected_rows = [{**row, "purpose": chosen[row["photo_id"]]}
                 for row in candidate_rows if row["photo_id"] in chosen]
main_rows = [row for row in selected_rows if row["purpose"] == "main"]
worker_rows = [row for row in selected_rows if row["purpose"] == "worker_train_exam"]
if (len(main_rows), len(worker_rows)) != (400, 100):
    raise ValueError("Нарушены размеры основной и учебной выборок")

contest_ids = set(contest)
contest_urls = set(contest.values())
main_ids = {r["photo_id"] for r in main_rows}
worker_ids = {r["photo_id"] for r in worker_rows}
main_urls = {r["photo_url"] for r in main_rows}
worker_urls = {r["photo_url"] for r in worker_rows}
if (len(main_ids), len(worker_ids), len(main_urls), len(worker_urls)) != (400, 100, 400, 100):
    raise ValueError("В выбранных данных повторяются ID или URL")
if (contest_ids & (main_ids | worker_ids)
        or contest_urls & (main_urls | worker_urls)
        or main_ids & worker_ids or main_urls & worker_urls):
    raise ValueError("Найдено пересечение с Контестом или между двумя выборками")
url_pattern = re.compile(r"https://inaturalist-open-data[.]s3[.]amazonaws[.]com/photos/([0-9]+)/original[.](?:jpg|jpeg|png)")
for row in selected_rows:
    match = url_pattern.fullmatch(row["photo_url"])
    if not match or int(match.group(1)) != row["photo_id"] or row["license_code"] != "cc0":
        raise ValueError(f"Фото {row['photo_id']}: некорректный URL оригинала или лицензия")
for group in groups:
    for purpose, expected in (("main", group["target_count"]),
                              ("worker_train_exam", group["training_exam_count"])):
        actual = sum(row["source_stratum"] == group["name"] and row["purpose"] == purpose
                     for row in selected_rows)
        if actual != expected:
            raise ValueError(f"Квота {group['name']}/{purpose}: {actual} вместо {expected}")

write_csv(candidate_rows, chosen, run_dir / "selection_500_proposal.csv")
write_gallery(candidate_rows, chosen, plan, run_dir / "candidate_gallery_500.html")
print("Выбрано: основная =", len(main_rows), "; материалы исполнителей =", len(worker_rows))
print("Пересечений по ID и URL с Контестом: 0")

## 3. Скачать и проверить оригиналы

Для выбранных 500 фото ноутбук проверяет размер изображения и SHA-256. Оригиналы сохраняются в `images/`. За один запуск загружается не более 1 ГиБ, размер одного файла ограничен 20 МиБ. Если интернет или оригинал недоступен, ячейка остановится, а JSONL для разметки не будут записаны.

In [ ]:
from scripts.download_selected import download, image_path, verify_image, checksum

image_dir = root / "images"
image_dir.mkdir(exist_ok=True)
max_new_bytes = 1024 * 1024 * 1024
max_file_bytes = 20 * 1024 * 1024
new_bytes = 0
audit_rows = []
hashes = {}
for index, row in enumerate(selected_rows, 1):
    target = image_path(row, image_dir)
    if not target.exists():
        download(row, target, max_file_bytes, max_new_bytes - new_bytes)
        new_bytes += target.stat().st_size
    verify_image(target, row["image_width"], row["image_height"])
    digest = checksum(target)
    if digest in hashes:
        raise ValueError(f"Файлы {hashes[digest]} и {row['photo_id']} побайтово одинаковы")
    hashes[digest] = row["photo_id"]
    audit_rows.append({**row, "local_path": str(target.relative_to(root)),
                       "bytes": target.stat().st_size, "sha256": digest})
    if index % 25 == 0 or index == len(selected_rows):
        print(f"Проверено {index}/{len(selected_rows)}; новых загрузок {new_bytes / 1024**2:.1f} МиБ")
write_jsonl(run_dir / "download_audit.jsonl", audit_rows)
print("Все 500 оригиналов проверены. Побайтовых дубликатов нет.")

## 4. Отметить визуально похожие пары

Сравнение dHash отмечает возможные дубликаты для ручной проверки. Само по себе малое расстояние хешей не доказывает, что фото одинаковые; поэтому программа не удаляет снимки автоматически.

In [ ]:
from scripts.audit_similarity import dhash

image_hashes = [(row["photo_id"], row["purpose"], dhash(root / row["local_path"]))
                for row in audit_rows]
similar_pairs = []
for index, (left_id, left_purpose, left_hash) in enumerate(image_hashes):
    for right_id, right_purpose, right_hash in image_hashes[index + 1:]:
        distance = (left_hash ^ right_hash).bit_count()
        if distance <= 4:
            similar_pairs.append((left_id, right_id, distance, left_purpose, right_purpose))
with (run_dir / "similarity_review.csv").open("w", encoding="utf-8", newline="") as target:
    writer = csv.writer(target)
    writer.writerow(["photo_id_a", "photo_id_b", "dhash_distance", "purpose_a", "purpose_b", "same_photo_yes_no", "notes"])
    for pair in similar_pairs:
        writer.writerow([*pair, "", ""])
print("Визуально похожих пар для ручной проверки:", len(similar_pairs))

## 5. Записать два списка и отчёт

В каждом JSONL только ID фотографии и публичная ссылка на оригинал. Обучающие, экзаменационные и контрольные задания требуют **проверенных ответов**; эти 100 фото пока лишь отдельный резерв и не делятся на роли автоматически.

In [ ]:
main_manifest = [{"photo_id": row["photo_id"], "photo_url": row["photo_url"]}
                 for row in main_rows]
worker_manifest = [{"photo_id": row["photo_id"], "photo_url": row["photo_url"]}
                   for row in worker_rows]
main_path = run_dir / "main_dataset_draft.jsonl"
worker_path = run_dir / "worker_training_exam_draft.jsonl"
write_jsonl(main_path, main_manifest)
write_jsonl(worker_path, worker_manifest)
if (len(read_jsonl(main_path)), len(read_jsonl(worker_path))) != (400, 100):
    raise RuntimeError("После записи размер JSONL не совпал с планом")
report = {
    "status": "technical_checks_passed; visual_review_pending",
    "retrieved_at": retrieved_at,
    "seed": seed,
    "plan_sha256": sha256_file(plan_path),
    "contest_sha256": sha256_file(contest_path),
    "candidate_count": len(candidate_rows),
    "main_count": len(main_manifest),
    "worker_count": len(worker_manifest),
    "main_by_source_stratum": dict(Counter(r["source_stratum"] for r in main_rows)),
    "worker_by_source_stratum": dict(Counter(r["source_stratum"] for r in worker_rows)),
    "contest_id_overlap": 0,
    "contest_url_overlap": 0,
    "main_worker_id_overlap": 0,
    "main_worker_url_overlap": 0,
    "new_download_bytes": new_bytes,
    "perceptual_similarity_pairs_for_review": len(similar_pairs),
    "main_sha256": sha256_file(main_path),
    "worker_sha256": sha256_file(worker_path),
}
(run_dir / "sampling_audit.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("Готово:")
print(" ", main_path)
print(" ", worker_path)
print(" ", run_dir / "sampling_audit.json")
print("Статус: черновик до визуального просмотра и фиксации списка.")